# Stage 6 cross-encoder reranker cache (Private)

Dataset input wajib **Private** karena memuat pasangan pertanyaan dan chunk klinis. Output hanya memuat ID hash, skor, dan manifest.

In [ ]:
import platform, socket, subprocess, sys
print('python', sys.version)
print('platform', platform.platform())
subprocess.run(['nvidia-smi'], check=True)
socket.create_connection(('huggingface.co', 443), timeout=10).close()
print('GPU_AND_INTERNET_OK')

In [ ]:
%pip install -q --upgrade transformers==5.19.0 huggingface-hub==1.33.0

In [ ]:
import hashlib, json, time
from pathlib import Path
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
ROOT = Path('/kaggle/input')
matches = sorted(ROOT.rglob('stage6_reranker_input.jsonl'))
manifests = sorted(ROOT.rglob('manifest.json'))
assert len(matches) == 1, matches
input_path = matches[0]
source_manifest = next(json.loads(path.read_text(encoding='utf-8')) for path in manifests if 'unique_pairs' in json.loads(path.read_text(encoding='utf-8')))
assert hashlib.sha256(input_path.read_bytes()).hexdigest() == source_manifest['input_sha256']
rows = [json.loads(line) for line in input_path.read_text(encoding='utf-8').splitlines() if line]
assert len(rows) == len({row['pair_id'] for row in rows}) == source_manifest['unique_pairs']
PRIMARY = (source_manifest['model_id'], source_manifest['revision'])
FALLBACK = (source_manifest['fallback_model_id'], source_manifest['fallback_revision'])
def load_model(spec):
    model_id, revision = spec
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, revision=revision, trust_remote_code=True, torch_dtype=torch.float16).to('cuda').eval()
    return tokenizer, model
fallback_used = False
try:
    tokenizer, model = load_model(PRIMARY)
    active = PRIMARY
except Exception as error:
    print('PRIMARY_LOAD_FAILED', repr(error))
    torch.cuda.empty_cache()
    tokenizer, model = load_model(FALLBACK)
    active, fallback_used = FALLBACK, True
print({'model_id': active[0], 'revision': active[1], 'fallback_used': fallback_used})

In [ ]:
BATCH_SIZE = 16
scores = []
started = time.perf_counter()
with torch.inference_mode():
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        encoded = tokenizer([[row['question'], row['passage']] for row in batch], padding=True, truncation=True, max_length=512, return_tensors='pt')
        encoded = {key: value.to('cuda') for key, value in encoded.items()}
        logits = model(**encoded).logits.reshape(-1).float().cpu().numpy()
        scores.extend(float(value) for value in logits)
        if start % (BATCH_SIZE * 200) == 0:
            print({'completed': min(start + len(batch), len(rows)), 'total': len(rows), 'elapsed_seconds': time.perf_counter() - started}, flush=True)
elapsed = time.perf_counter() - started
assert len(scores) == len(rows) and np.isfinite(scores).all()
print({'pairs': len(scores), 'elapsed_seconds': elapsed, 'gpu': torch.cuda.get_device_name(0)})

In [ ]:
import shutil
OUT = Path('/kaggle/working/stage6_reranker_output')
OUT.mkdir(exist_ok=True)
score_path = OUT / 'scores.jsonl'
with score_path.open('w', encoding='utf-8', newline='\n') as handle:
    for row, score in zip(rows, scores):
        handle.write(json.dumps({'pair_id': row['pair_id'], 'score': score}) + '\n')
def digest(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
manifest = {'model_id': active[0], 'revision': active[1], 'fallback_used': fallback_used, 'input_sha256': source_manifest['input_sha256'], 'pairs': len(rows), 'batch_size': BATCH_SIZE, 'max_length': 512, 'elapsed_seconds': elapsed, 'device': torch.cuda.get_device_name(0), 'scores_sha256': digest(score_path)}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive('/kaggle/working/stage6_reranker_output', 'zip', OUT)
print(json.dumps(manifest, indent=2))
print('OUTPUT_ARCHIVE', archive)